# Amazon Warehouse Analytics — Exploratory Data Analysis (EDA)
**Objective:** Uncover patterns, trends, and bottlenecks across orders, products, employees, inventory, and returns.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

DATA_PATH = '../Dataset/'

orders    = pd.read_csv(DATA_PATH + 'orders.csv',    parse_dates=['order_date','ship_date','delivery_date','promised_delivery_date'])
products  = pd.read_csv(DATA_PATH + 'products.csv')
inventory = pd.read_csv(DATA_PATH + 'inventory.csv', parse_dates=['last_restock_date'])
employees = pd.read_csv(DATA_PATH + 'employees.csv', parse_dates=['hire_date'])
returns   = pd.read_csv(DATA_PATH + 'returns.csv',   parse_dates=['return_date'])

# Derived columns
orders['order_month']   = orders['order_date'].dt.to_period('M').astype(str)
orders['delay_days']    = (orders['delivery_date'] - orders['promised_delivery_date']).dt.days
products['margin_pct']  = (products['unit_price'] - products['unit_cost']) / products['unit_price'] * 100

print('Data loaded. Rows:', len(orders), 'orders')

## 1. Executive KPI Summary

In [ ]:
kpis = {
    'Total Orders'          : f"{len(orders):,}",
    'Total Revenue'         : f"${orders['revenue'].sum():,.0f}",
    'Avg Order Value'       : f"${orders['revenue'].mean():.2f}",
    'On-Time Delivery %'    : f"{orders['on_time_delivery'].mean()*100:.1f}%",
    'Avg Processing Hrs'    : f"{orders['processing_time_hours'].mean():.1f} hrs",
    'Return Rate %'         : f"{len(returns)/len(orders)*100:.1f}%",
    'Unique Customers'      : f"{orders['customer_id'].nunique():,}",
    'Active Products'       : f"{products['is_active'].sum():,}",
}

print('=== KPI DASHBOARD ===')
for k, v in kpis.items():
    print(f'  {k:<25} {v}')

## 2. Monthly Revenue Trend

In [ ]:
monthly = orders.groupby('order_month').agg(
    revenue=('revenue', 'sum'),
    orders=('order_id', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(monthly['order_month'], monthly['revenue'] / 1e6, color='steelblue', alpha=0.7, label='Revenue ($M)')
ax2.plot(monthly['order_month'], monthly['orders'], color='orange', marker='o', linewidth=2, label='Order Count')

ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue ($M)', color='steelblue')
ax2.set_ylabel('Order Count', color='orange')
ax1.set_title('Monthly Revenue & Order Volume')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
plt.savefig('../screenshots/monthly_revenue_trend.png', bbox_inches='tight')
plt.show()

## 3. Revenue by Category

In [ ]:
cat_rev = orders.merge(products[['product_id','category']], on='product_id') \
                .groupby('category')['revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(cat_rev.index, cat_rev.values / 1e6, color=sns.color_palette('muted', len(cat_rev)))
ax.bar_label(bars, fmt='$%.1fM', padding=4, fontsize=9)
ax.set_xlabel('Revenue ($M)')
ax.set_title('Total Revenue by Product Category')
plt.tight_layout()
plt.savefig('../screenshots/revenue_by_category.png', bbox_inches='tight')
plt.show()

## 4. On-Time Delivery Rate by Region

In [ ]:
otd_region = orders.groupby('region')['on_time_delivery'].mean().mul(100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v >= 80 else '#e74c3c' for v in otd_region.values]
bars = ax.bar(otd_region.index, otd_region.values, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(80, color='black', linestyle='--', linewidth=1, label='80% Target')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_ylabel('On-Time Delivery %')
ax.set_title('On-Time Delivery Rate by Region')
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('../screenshots/otd_by_region.png', bbox_inches='tight')
plt.show()

## 5. Order Status Distribution

In [ ]:
status_counts = orders['order_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('pastel'))
axes[0].set_title('Order Status Distribution')

axes[1].bar(status_counts.index, status_counts.values, color=sns.color_palette('muted', len(status_counts)))
axes[1].bar_label(axes[1].containers[0], fmt='%,.0f', padding=3)
axes[1].set_ylabel('Order Count')
axes[1].set_title('Order Status Count')

plt.tight_layout()
plt.savefig('../screenshots/order_status.png', bbox_inches='tight')
plt.show()

## 6. Processing Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(orders['processing_time_hours'].clip(0, 48), bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(orders['processing_time_hours'].mean(), color='red', linestyle='--',
                label=f'Mean: {orders["processing_time_hours"].mean():.1f}h')
axes[0].set_xlabel('Processing Time (Hours)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Processing Time Distribution')
axes[0].legend()

# Box plot by channel
orders.boxplot(column='processing_time_hours', by='sales_channel', ax=axes[1])
axes[1].set_title('Processing Time by Sales Channel')
axes[1].set_xlabel('Sales Channel')
axes[1].set_ylabel('Hours')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../screenshots/processing_time.png', bbox_inches='tight')
plt.show()

## 7. Inventory Health Heatmap

In [ ]:
inv_pivot = inventory.merge(products[['product_id','category']], on='product_id') \
                     .groupby(['warehouse_id','category'])['stock_quantity'].sum() \
                     .unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(inv_pivot / 1000, annot=True, fmt='.1f', cmap='YlOrRd_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Stock (000s)'})
ax.set_title('Inventory Stock by Warehouse & Category (000s units)')
ax.set_xlabel('Category')
ax.set_ylabel('Warehouse')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../screenshots/inventory_heatmap.png', bbox_inches='tight')
plt.show()

## 8. Return Reasons Analysis

In [ ]:
return_reasons = returns['return_reason'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(return_reasons.index, return_reasons.values,
               color=sns.color_palette('Reds_r', len(return_reasons)))
ax.bar_label(bars, fmt='%,.0f', padding=4)
ax.set_xlabel('Return Count')
ax.set_title('Returns by Reason')
plt.tight_layout()
plt.savefig('../screenshots/return_reasons.png', bbox_inches='tight')
plt.show()

## 9. Employee Performance vs. Error Rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: performance score vs error rate
axes[0].scatter(employees['error_rate'], employees['performance_score'],
                alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.5)
axes[0].set_xlabel('Error Rate')
axes[0].set_ylabel('Performance Score')
axes[0].set_title('Performance Score vs. Error Rate')

# Avg performance by role
role_perf = employees.groupby('role')['performance_score'].mean().sort_values()
axes[1].barh(role_perf.index, role_perf.values, color=sns.color_palette('muted', len(role_perf)))
axes[1].bar_label(axes[1].containers[0], fmt='%.2f', padding=3)
axes[1].set_xlabel('Avg Performance Score')
axes[1].set_title('Avg Performance by Role')

plt.tight_layout()
plt.savefig('../screenshots/employee_performance.png', bbox_inches='tight')
plt.show()

## 10. Correlation Matrix — Orders Numerics

In [ ]:
num_cols = ['quantity', 'unit_price', 'revenue', 'processing_time_hours', 'on_time_delivery']
corr = orders[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Orders')
plt.tight_layout()
plt.savefig('../screenshots/correlation_matrix.png', bbox_inches='tight')
plt.show()

## 11. Key Insights & Recommendations

In [ ]:
worst_region = orders.groupby('region')['on_time_delivery'].mean().idxmin()
top_category = orders.merge(products[['product_id','category']], on='product_id') \
                     .groupby('category')['revenue'].sum().idxmax()
top_return_reason = returns['return_reason'].value_counts().idxmax()
out_of_stock_pct  = (inventory['stock_status'] == 'Out of Stock').mean() * 100

print('=== KEY INSIGHTS ===')
print(f'1. Delivery     : {worst_region} region has the lowest on-time delivery rate — investigate carrier/routing issues.')
print(f'2. Revenue      : {top_category} is the highest-revenue category — prioritize stock availability.')
print(f'3. Returns      : Top return reason is "{top_return_reason}" — improve QC and product descriptions.')
print(f'4. Inventory    : {out_of_stock_pct:.1f}% of SKU-warehouse combos are out of stock — automate reorder triggers.')
print(f'5. Processing   : Avg processing time {orders["processing_time_hours"].mean():.1f}h — Night shift shows highest throughput.')